# CelLab, Deliverable 3: training on Google Colab

This notebook trains and evaluates the networks of the experiment. Everything else (the datasets, the
models, the training and evaluation code) is in the repository; the notebook only runs it on a GPU.

**Before starting:** *Runtime → Change runtime type → T4 GPU*. Then run the cells from top to bottom.
Results are written to your Google Drive, so nothing is lost if the session ends: run the cells again
and training continues where it stopped.

| Part | What it does | Time on a T4 (roughly) |
|---|---|---|
| 1 | Set-up: code, data, Drive | 2 min |
| 2 | One-step network on the 256 elementary rules | 5 min |
| 3 | Five-questions network (one pass), then five separate networks | 40 min |
| 4 | The 2D automata | 30 min |
| 5 | Summary tables and pictures | 1 min |

## 1. Set-up

In [ ]:
REPO = "https://github.com/driedpampas/cellab.git"
BRANCH = "main"
SAVE_TO_DRIVE = True          # False: keep results only in this session

import os, subprocess, sys
if not os.path.isdir("/content/cellab"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/cellab"], check=True)
os.chdir("/content/cellab")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

RESULTS = "/content/results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/cellab_results"
os.makedirs(RESULTS, exist_ok=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - choose a GPU runtime!")
print("results go to", RESULTS)

The datasets are in `data/` in the repository. If one is missing (or you changed a file in `datasets/`),
this cell generates it again; the result is identical, because every sample has its own seed.

In [ ]:
import glob
for spec in sorted(glob.glob("datasets/*.json")):
    name = os.path.basename(spec)[:-5]
    if not os.path.exists(f"data/{name}.npz"):
        print("generating", name)
        subprocess.run(["cellab", "dataset", spec, "--workers", "2"], check=True)
print(len(glob.glob("data/*.npz")), "datasets ready")

In [ ]:
def run(*args):
    """Run one of the project's commands and show its output as it comes."""
    print(">", " ".join(args))
    subprocess.run([sys.executable, "-m", *args], check=True)

def train_and_evaluate(kind, dataset, name, epochs, extra=(), also=()):
    out = f"{RESULTS}/{name}"
    run("ml.train", kind, f"data/{dataset}.npz", "--out", out, "--epochs", str(epochs), "--resume", *extra)
    run("ml.evaluate", kind, f"data/{dataset}.npz", f"{out}/best.pt", "--out", out)
    for other in also:          # the same trained model on another dataset
        run("ml.evaluate", kind, f"data/{other}.npz", f"{out}/best.pt", "--out", f"{out}_on_{other}")

## 2. The one-step network

It learns a single step, `row + rule → next row`, for all 256 rules at once, and answers "after N steps"
by applying itself N times. 15% of the rule families (Rule 30's among them) are never shown in training.
The same trained network is then tested on rows of 32 cells, twice as wide as anything it was trained on.

In [ ]:
train_and_evaluate("step", "elementary_w16", "step_elementary", epochs=40, also=["elementary_w32_test"])

## 3. The five-questions network

One pass, no simulation: the row, the rule, N and a configuration C go in, and five heads answer.
First one shared network, then five networks that each learn a single question (the comparison
promised in Deliverable 1: one network or separate ones?).

In [ ]:
train_and_evaluate("answers", "elementary_w16", "answers_elementary", epochs=80, also=["elementary_w32_test"])

In [ ]:
for task in ("state", "count", "stable", "cycle", "occurs"):
    train_and_evaluate("answers", "elementary_w16", f"only_{task}_elementary", epochs=80, extra=["--tasks", task])

## 4. The 2D automata

One model per automaton. The one-step network is trained where a step is fully determined by the grid
(deterministic automata without layers); the five-questions network is trained for all of them.
For automata with random rules its targets are single random outcomes, so compare it with the baselines
rather than expecting exact grids.

In [ ]:
import json
TWO_D = sorted(os.path.basename(p)[:-5] for p in glob.glob("data/*_16.json") + glob.glob("data/traffic_w16.json"))
for name in TWO_D:
    info = json.load(open(f"data/{name}.json"))
    if info["arrays"]["states"]["type"].startswith("float"):
        print("skipping", name, "(decimal values)")
        continue
    if info["automaton"]["deterministic"] and not info["automaton"]["layers"]:
        train_and_evaluate("step", name, f"step_{name}", epochs=30)
    train_and_evaluate("answers", name, f"answers_{name}", epochs=40)

## 5. Summary

In [ ]:
import pandas as pd
rows = []
for path in sorted(glob.glob(f"{RESULTS}/*/metrics.json")):
    report = json.load(open(path))
    for split, groups in report["splits"].items():
        for group, scores in groups.items():
            if "by_horizon" not in scores:
                continue
            row = {"model": os.path.basename(os.path.dirname(path)), "split": split, "group": group}
            for h in scores["by_horizon"]:
                row[f"cells N={h['n']}"] = round(h["model"]["cells_correct"], 3)
            last = scores["by_horizon"][-1]
            row["grids (largest N)"] = round(last["model"]["grids_correct"], 3)
            row["copy baseline (largest N)"] = round(last["copy"]["cells_correct"], 3)
            row["count error (largest N)"] = round(last["model"]["count_error"], 2)
            for key in ("stable", "occurs"):
                if key in scores:
                    row[f"{key} accuracy"], row[f"{key} F1"] = round(scores[key]["accuracy"], 3), round(scores[key]["f1"], 3)
            for key in ("cycle_start", "cycle_length"):
                if key in scores:
                    row[f"{key} error"], row[f"{key} common error"] = round(scores[key]["error"], 2), round(scores[key]["common_error"], 2)
            rows.append(row)
table = pd.DataFrame(rows)
table.to_csv(f"{RESULTS}/summary.csv", index=False)
pd.set_option("display.max_rows", 500, "display.width", 250)
table[table.group == "all"]

In [ ]:
# The main result: the elementary rules by Wolfram class, for both kinds of network.
table[table.model.isin(["step_elementary", "answers_elementary"])].sort_values(["split", "group", "model"])

In [ ]:
from IPython.display import Image, display
for name in ("step_elementary", "answers_elementary", "step_elementary_on_elementary_w32_test"):
    picture = f"{RESULTS}/{name}/accuracy_by_horizon.png"
    if os.path.exists(picture):
        print(name); display(Image(picture))

Everything is now in the results folder (on your Drive if `SAVE_TO_DRIVE` was on): for every model
`best.pt` (the trained network), `history.json` (every epoch), `metrics.json` (all numbers) and
`accuracy_by_horizon.png`; and `summary.csv` with the tables above. Download the folder and put it in
`results/` in the repository to use the numbers in the Deliverable 3 presentation.